# Forecast Discounted Method

This notebook will focus on a couple of ideas. The first part of this notebook is to evaluate prediction quality of various models specifically models that extend beyond the OLS framework. All these models’ goal is to predict End of Quarter values. The second part of this notebook is turning that end of quarter values into daily values and testing the exposure and statistical significance of those factors. Currently the main test is to compare the use of MIDAS (Mixed-data Sampling) Regressions relative to OLS. Note that all fundamental values are differenced and therefore it should be assumed so throughout this future. 

# Packages

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from   sklearn.metrics import root_mean_squared_error

import matplotlib.pyplot as plt
from   matplotlib.ticker import FuncFormatter

In [2]:
from tqdm import tqdm
tqdm.pandas()

In [3]:
note_path  = os.getcwd()
repo_path  = os.path.abspath(os.path.join(note_path, ".."))
src_path   = os.path.join(repo_path, "src")
data_path  = os.path.join(repo_path, "data")
db_path    = os.path.join(data_path, "DataBase")
quart_path = os.path.join(db_path, "Quarterlyized")
daily_path = os.path.join(db_path, "DailyData")
model_path = os.path.join(data_path, "Models")
base_path  = os.path.join(model_path, "BaselineModels")

In [4]:
import sys
sys.path.append(src_path)
from OLSModels import OLSModels

In [5]:
sample_tickers = ['SPX Index', 'RTY Index', 'NDX Index']

In [6]:
ticker_path = os.path.join(data_path, "EquityForecastTickerGuide.xlsx")
df_renamer  = pd.read_excel(io = ticker_path, sheet_name = "Renamer")

RENAMER = (df_renamer
    .set_index("old_name")
    .loc[lambda x: x.group == "renamer"]
    .new_name
    .to_dict())

PLOT_RENAMER = (df_renamer
    .set_index("old_name")
    .loc[lambda x: x.group == "plot_renamer"]
    .assign(name = lambda x: x.new_name.str.replace(r"\\n", "\n", regex = True))
    .name
    .to_dict())

FLOAT_FORMAT  = "{:.2f}".format
FLOAT_FORMAT4 = "{:.4f}".format

# Comparing OLS and MIDAS Prediction

At this current version all regressions are ran full-sample and in-sample. We have the following regression we'd like to compare each adding an extra component
1. OLS Model 1: Using lagged fundamental values to predict future fundamental values
2. OLS Model 2: Using lagged fundamentals and retunrs to predict future fundamental values
3. MIDAS Model 1: Using lagged fundamental values and daily returns with the restricted model
4. MIDAS Model 2: Using lagged fundamental values and daily returns with the unrestricted model 

In [7]:
raw_path = r"A:\GitHub\CurrentFundamentalFactorModels\data\Models\BaselineModels"

In [8]:
path = os.path.join(raw_path, "SingleVarPredictiveModels.pkl")
with open(path, "rb") as f: single_models = pickle.load(f)

path = os.path.join(raw_path, "MultiVarPredictiveModels.pkl")
with open(path, "rb") as f: multi_models = pickle.load(f)

In [9]:
def _get_ols_rsquared(models: dict) -> pd.DataFrame:

    df_out = (pd.concat([
        pd.DataFrame.from_dict(
            data   = {
                "model_name": [model_name], 
                "r_squared" : [models[model_name].rsquared],
                "aic"       : [models[model_name].aic]},
            orient = "columns") 
        for model_name in models.keys()]))

    return df_out

In [10]:
df_ols_rsquared = (pd.concat([
    _get_ols_rsquared(single_models).assign(model = "single"),
    _get_ols_rsquared(multi_models).assign(model = "multi")])
    .assign(
        str_split = lambda x: x.model_name.str.split("-"),
        ticker    = lambda x: x.str_split.str[0],
        field     = lambda x: x.str_split.str[1]))

The difference in MIDAS R2 between using vol-targeted returns or not is very small, only adding a small amount, therefore for brevity we'll use volatility adjusted returns. To keep the model simple we'll look just at SPY

In [29]:
path = r"A:\GitHub\CurrentFundamentalFactorModels\data\Models\MIDASModels\MIDASResults.parquet"
df_aic = (pd
    .read_parquet(path = path, engine = "pyarrow")
    .drop(columns = ["restricted_r2", "unrestricted_r2", "delta_aic"])
    .melt(id_vars = ["field", "ticker", "vol_target"], var_name = "model"))

In [42]:
df_midas_aic = (df_aic
    .loc[lambda x: x.vol_target == "vol_rtn"]
    .loc[lambda x: x.ticker == "SPX Index"]
    .pivot(index = "field", columns = "model", values = "value")
    .rename(columns = {
        "restricted_aic"  : "resitricted_midas",
        "unrestricted_aic": "unresitricted_midas"}))

In [44]:
df_ols_aic = (df_ols_rsquared
    .loc[lambda x: x.ticker == "SPX"]
    .pivot(index = "field", columns = "model", values = "aic")
    .rename(columns = {
        "multi" : "multi_ols",
        "single": "single_ols"}))

In [48]:
display(df_midas_aic
    .merge(right = df_ols_aic, how = "outer", on = ["field"]))

model,resitricted_midas,unresitricted_midas,multi_ols,single_ols
field,,,,
CURRENT_EV_TO_T12M_EBITDA,515.667300,553.456189,467.933490,124.244206
CURR_ENTP_VAL,1894.498157,1928.691398,1828.146794,1593.046363
EBIT,1409.626094,1431.282744,1304.927942,642.513855
EBITDA,999.950762,1026.331245,974.725704,766.097895
EV_TO_T12M_EBITDA,491.500603,543.967866,400.143290,125.612849
FREE_CASH_FLOW_PER_SH,1084.271029,1130.777757,1050.548157,591.765623
GROSS_MARGIN,343.869939,390.243447,336.898430,-237.218656
IS_EPS,1191.612365,1254.362233,1146.037843,668.340375


$\uparrow$ check MIDAS AIC

In [69]:
df_midas_r2 = (pd
    .read_parquet(path = path, engine = "pyarrow")
    .loc[lambda x: x.vol_target == "vol_rtn"]
    .loc[lambda x: x.ticker == "SPX Index"]
    [["field", "restricted_r2", "unrestricted_r2"]])

In [64]:
df_ols_r2 = (df_ols_rsquared
    .loc[lambda x: x.ticker == "SPX"]
    .pivot(index = "field", columns = "model", values = "r_squared")
    .add_suffix("_ols"))

When the $R^2$ a

In [71]:
(df_midas_r2
    .merge(right = df_ols_r2, how = "inner", on = ["field"])
    .rename(columns = {
        "restricted_r2"  : "restricted_midas",
        "unrestricted_r2": "unrestricted_midas"}))

,field,restricted_midas,unrestricted_midas,multi_ols,single_ols
0,CURRENT_EV_TO_T12M_EBITDA,0.145223,0.510354,0.364358,0.003117
1,CURR_ENTP_VAL,0.111231,0.499093,0.011879,0.364607
2,EBIT,0.168802,0.576328,0.106996,0.422507
3,EBITDA,0.168680,0.564093,0.176302,0.185681
4,EV_TO_T12M_EBITDA,0.131299,0.456685,0.543154,0.003277
5,FREE_CASH_FLOW_PER_SH,0.082258,0.451513,0.316187,0.073825
6,GROSS_MARGIN,0.060633,0.431289,0.096800,0.025411
7,IS_EPS,0.344328,0.551779,0.317419,0.266853
